In [1]:
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/datasets/ngocccchannn/full-interactions/interactions_full.parquet
/kaggle/input/datasets/ngocccchannn/cold-scenario/cold_items_test.json
/kaggle/input/datasets/ngocccchannn/cold-scenario/cold_items_val.json


In [2]:
import os
import shutil
import json
import numpy as np
import pandas as pd
from scipy.stats import spearmanr
import torch
from IPython.display import FileLink

print("Đã khởi tạo xong môi trường!")

Đã khởi tạo xong môi trường!


In [3]:
inter = pd.read_parquet('/kaggle/input/datasets/ngocccchannn/full-interactions/interactions_full.parquet')
print(inter.shape)
inter.head()

(11522940, 3)


,user_id,course_id,enroll_time
0,U_9137692,C_584313,1557326901
1,U_9137692,C_584329,1557327704
2,U_361418,C_584329,1557328370
3,U_69,C_584329,1557328556
4,U_69,C_584313,1557329610


In [4]:
# Lấy mẫu 100.000 học viên và chia lại với mốc cố định
SEED, N_USERS = 2026, 100_000 

# 1) Cố định mốc thời gian và danh sách khóa lạnh từ TOÀN BỘ dữ liệu 
t1, t2 = inter.enroll_time.quantile([0.8, 0.9]).values 

with open("/kaggle/input/datasets/ngocccchannn/cold-scenario/cold_items_val.json", "r", encoding="utf-8") as f:
    cold_val = set(json.load(f))  # 251 khóa 

with open("/kaggle/input/datasets/ngocccchannn/cold-scenario/cold_items_test.json", "r", encoding="utf-8") as f:
    cold_test = set(json.load(f)) # 502 khóa

print("Số cold items validation:", len(cold_val))
print("Số cold items test:", len(cold_test))
print('\n')

print('Dữ liệu ban đầu - trước khi lọc')
print(inter.shape)
print(f'user_unique: {inter['user_id'].nunique()}')
print(f'course_unique: {inter['course_id'].nunique()}')

# 2) Lấy mẫu đều học viên, không lọc k-core 
users = np.sort(inter.user_id.unique()) 
keep_u = set(np.random.default_rng(SEED).choice(users, size=N_USERS, replace=False)) 
df = inter[inter.user_id.isin(keep_u)].copy() 
print('Dữ liệu sau khi lọc 100.000 học viên')
print(df.shape)
print(f'user_unique: {df['user_id'].nunique()}')
print(f'course_unique: {df['course_id'].nunique()}')
  
# 3) Kịch bản Warm với t1, t2 cố định 
tr = df[df.enroll_time < t1] 
va = df[(df.enroll_time >= t1) & (df.enroll_time < t2)] 
te = df[df.enroll_time >= t2] 
ku, ki = set(tr.user_id), set(tr.course_id) 
va = va[va.user_id.isin(ku) & va.course_id.isin(ki)] 
te = te[te.user_id.isin(ku) & te.course_id.isin(ki)] 

print(f'train: {tr.shape}')
print(f'val: {va.shape}')
print(f'test: {te.shape}')

# 4) Kịch bản Cold-item với danh sách khóa lạnh cố định 
tr_c = df[~df.course_id.isin(cold_val | cold_test)] 
va_c = df[df.course_id.isin(cold_val)  & df.user_id.isin(tr_c.user_id)] 
te_c = df[df.course_id.isin(cold_test) & df.user_id.isin(tr_c.user_id)] 
assert not (set(tr_c.course_id) & (cold_val | cold_test)) 
print("Khóa cold-test còn tương tác:", te_c.course_id.nunique(), "/", len(cold_test)) 
print(f'train_col: {tr_c.shape}')
print(f'val_col: {va_c.shape}')
print(f'test_col: {te_c.shape}')

Số cold items validation: 251
Số cold items test: 502


Dữ liệu ban đầu - trước khi lọc
(11522940, 3)
user_unique: 3211039
course_unique: 3769
Dữ liệu sau khi lọc 100.000 học viên
(357139, 3)
user_unique: 100000
course_unique: 2861
train: (286816, 3)
val: (21833, 3)
test: (22437, 3)
Khóa cold-test còn tương tác: 268 / 502
train_col: (346659, 3)
val_col: (3053, 3)
test_col: (5531, 3)


In [5]:
# Kiểm tra phân phối độ phổ biến trước/sau lấy mẫu 
pf, ps = inter.course_id.value_counts(), df.course_id.value_counts() 
common = pf.index.intersection(ps.index) 
print("Spearman độ phổ biến:", spearmanr(pf[common], ps[common]).correlation) 

Spearman độ phổ biến: 0.9934742151690644


In [6]:
# ===== Bảng thống kê và các điều kiện kiểm tra (Mục 6.3 bước 5 và 6) =====
assert len(cold_val) == 251 and len(cold_test) == 502, 'Danh sách khóa lạnh không còn 251 (val) / 502 (test) khóa như báo cáo'

def thong_ke(ten, d):
    return {'tập': ten, 'số học viên': d['user_id'].nunique(), 'số khóa': d['course_id'].nunique(), 'số tương tác': len(d)}

stats = pd.DataFrame([
    thong_ke('Toàn bộ dữ liệu', inter),
    thong_ke('Mẫu 100.000 học viên', df),
    thong_ke('Warm - train', tr), thong_ke('Warm - val', va), thong_ke('Warm - test', te),
    thong_ke('Cold-item - train', tr_c), thong_ke('Cold-item - val', va_c), thong_ke('Cold-item - test', te_c),
])
print(stats.to_string(index=False))
stats.to_csv('/kaggle/working/split_statistics.csv', index=False, encoding='utf-8')

# Số khóa cold-test còn ít nhất 1 tương tác trong mẫu
n_cold_left = te_c['course_id'].nunique()
ti_le_mat = 1 - n_cold_left / len(cold_test)
print(f"\nKhóa cold-test còn ít nhất 1 tương tác: {n_cold_left}/{len(cold_test)} (mất {ti_le_mat:.1%})")

# Mức tương quan độ phổ biến khóa học trước và sau lấy mẫu (gợi ý >= 0,95 [giả định ngưỡng])
rho = spearmanr(pf[common], ps[common]).correlation
print(f"Spearman độ phổ biến trước/sau lấy mẫu: {rho:.4f} ->", 'ĐẠT (>= 0,95)' if rho >= 0.95 else 'CHƯA ĐẠT ngưỡng gợi ý 0,95')

# Ước tính trong báo cáo: ~348.000 train, ~3.000 cold-val, ~5.600 cold-test
print(f"\nSo với ước tính trong báo cáo: train_cold = {len(tr_c):,} (~348.000) | val_cold = {len(va_c):,} (~3.000) | test_cold = {len(te_c):,} (~5.600)")
if len(te_c) < 3000 or ti_le_mat > 0.20:
    print('=> [CẢNH BÁO] cold-test dưới 3.000 tương tác hoặc hơn 20% khóa cold-test không còn tương tác: BÁO GVHD TRƯỚC KHI CHẠY TIẾP (Mục 6.3 bước 6).')
else:
    print('=> Đạt điều kiện ở Mục 6.3 bước 6, có thể chạy tiếp.')


                 tập  số học viên  số khóa  số tương tác
     Toàn bộ dữ liệu      3211039     3769      11522940
Mẫu 100.000 học viên       100000     2861        357139
        Warm - train        88729     2448        286816
          Warm - val         4948     1654         21833
         Warm - test         4712     1609         22437
   Cold-item - train        98220     2401        346659
     Cold-item - val         2092      145          3053
    Cold-item - test         3340      268          5531

Khóa cold-test còn ít nhất 1 tương tác: 268/502 (mất 46.6%)
Spearman độ phổ biến trước/sau lấy mẫu: 0.9935 -> ĐẠT (>= 0,95)

So với ước tính trong báo cáo: train_cold = 346,659 (~348.000) | val_cold = 3,053 (~3.000) | test_cold = 5,531 (~5.600)
=> [CẢNH BÁO] cold-test dưới 3.000 tương tác hoặc hơn 20% khóa cold-test không còn tương tác: BÁO GVHD TRƯỚC KHI CHẠY TIẾP (Mục 6.3 bước 6).


In [7]:
scen = "warm_scenario"
output_dir = os.path.join("/kaggle/working", scen)
os.makedirs(output_dir, exist_ok=True)

# Khai báo trực tiếp map giữa tên file và biến DataFrame
warm_splits = {
    "train_warm": tr,
    "val_warm": va,
    "test_warm": te
}

for name, part in warm_splits.items():
    part.to_parquet(os.path.join(output_dir, f"{name}.parquet"), index=False, engine="pyarrow")
    print(f"Đã lưu: {name}.parquet")

shutil.make_archive("/kaggle/working/warm_scenario", 'zip', "/kaggle/working/warm_scenario")
print("Đã nén xong!")

FileLink('warm_scenario.zip')

Đã lưu: train_warm.parquet
Đã lưu: val_warm.parquet
Đã lưu: test_warm.parquet
Đã nén xong!


/kaggle/working/warm_scenario.zip

In [8]:
scen = "cold_scenario"
output_dir = os.path.join("/kaggle/working", scen)
os.makedirs(output_dir, exist_ok=True)

# Khai báo trực tiếp map giữa tên file và biến DataFrame
warm_splits = {
    "train_cold": tr_c,
    "val_cold": va_c,
    "test_cold": te_c
}

for name, part in warm_splits.items():
    part.to_parquet(os.path.join(output_dir, f"{name}.parquet"), index=False, engine="pyarrow")
    print(f"Đã lưu: {name}.parquet")

shutil.make_archive("/kaggle/working/cold_scenario", 'zip', "/kaggle/working/cold_scenario")
print("Đã nén xong!")

FileLink('cold_scenario.zip')

Đã lưu: train_cold.parquet
Đã lưu: val_cold.parquet
Đã lưu: test_cold.parquet
Đã nén xong!


/kaggle/working/cold_scenario.zip

In [9]:
# 3. Chia dữ liệu thành 3 tập theo kịch bản 3 - FEW-SHOT-USER
def few_shot_user_split(df, t1, ks=(1, 3, 5), val_user_ratio=0.33, seed=42):
    """
    df: user_id, course_id, enroll_time (đã khử trùng cặp user-course).
    Trả về: tr, {k: (val_fold_in, val_target, test_fold_in, test_target)}
    """
    first_time = df.groupby('user_id')['enroll_time'].min()
    new_users = first_time.index[first_time >= t1]

    tr = df[df['enroll_time'] < t1].copy()
    known_items = set(tr['course_id'])

    # Chia val/test user MỘT LẦN cho mọi k (tập user lồng nhau giữa các k)
    users = np.sort(new_users.to_numpy())
    np.random.default_rng(seed).shuffle(users)
    n_val = int(len(users) * val_user_ratio)
    val_users, test_users = set(users[:n_val]), set(users[n_val:])

    # Chỉ giữ tương tác của user mới với các khóa học đã xuất hiện trong train
    df_new = df[df['user_id'].isin(new_users) & df['course_id'].isin(known_items)].copy()
    df_new = df_new.sort_values(['user_id', 'enroll_time', 'course_id'], kind='stable')
    df_new['pos'] = df_new.groupby('user_id').cumcount()
    
    cnt = df_new.groupby("user_id").size()  
    pop = set(cnt[cnt >= max(ks) + 1].index)

    def cut(sub, k):
        fold_in = sub[sub['pos'] < k].drop(columns='pos')
        target = sub[sub['pos'] >= k].drop(columns='pos')
        return fold_in, target

    out = {}
    print(f"--- THỐNG KÊ FEW-SHOT USERS (Mốc t1 = {t1}) ---")
    print(f"Số tương tác Train: {len(tr):,} | Số Items Train: {len(known_items):,}")
 
    for k in ks:
        sub = df_new[df_new.user_id.isin(pop)] 
        vf, vt = cut(sub[sub.user_id.isin(val_users)], k) 
        tf, tt = cut(sub[sub.user_id.isin(test_users)], k) 
        
        # Đảm bảo có ít nhất 1 user ở cả 2 tập
        assert len(vf) > 0 and len(tf) > 0, f"Không đủ người dùng cho k={k}!"

        for f, t in ((vf, vt), (tf, tt)):
            assert f.groupby('user_id').size().eq(k).all(), f"Fold-in phải đúng {k} tương tác/user"
            assert set(f.user_id) == set(t.user_id), "Mỗi user phải có cả fold-in lẫn target"
            assert set(t['course_id']) <= known_items, "Target chứa khóa chưa có trong train"
            assert not (set(zip(f.user_id, f.course_id)) & set(zip(t.user_id, t.course_id))), "Trùng fold-in/target"
        
        n_val_u = vf['user_id'].nunique()
        n_test_u = tf['user_id'].nunique()
        print(f" k = {k}: Hợp lệ {n_val_u:,} Val Users | {n_test_u:,} Test Users (Target items: Val={len(vt):,}, Test={len(tt):,})")
        
        out[k] = (vf, vt, tf, tt)

    assert not (set(tr['user_id']) & (val_users | test_users)), "User mới lọt vào train"
    assert not (val_users & test_users), "User vừa val vừa test"
    
    return tr, out

train_df, few_shot_dict = few_shot_user_split(df, t1, ks=(1, 3, 5), seed=42)

--- THỐNG KÊ FEW-SHOT USERS (Mốc t1 = 1597169939.2) ---
Số tương tác Train: 286,816 | Số Items Train: 2,448
 k = 1: Hợp lệ 154 Val Users | 328 Test Users (Target items: Val=1,585, Test=3,375)
 k = 3: Hợp lệ 154 Val Users | 328 Test Users (Target items: Val=1,277, Test=2,719)
 k = 5: Hợp lệ 154 Val Users | 328 Test Users (Target items: Val=969, Test=2,063)


In [10]:
scen_base = "/kaggle/working/few_shot_splits"
os.makedirs(scen_base, exist_ok=True)

train_df.to_parquet(os.path.join(scen_base, "train.parquet"), index=False, engine="pyarrow")
print("Đã lưu: train.parquet")

for k, (vf, vt, tf, tt) in few_shot_dict.items():
    k_dir = os.path.join(scen_base, f"k_{k}")
    os.makedirs(k_dir, exist_ok=True)
    
    vf.to_parquet(os.path.join(k_dir, "val_fold_in.parquet"), index=False)
    vt.to_parquet(os.path.join(k_dir, "val_target.parquet"), index=False)
    tf.to_parquet(os.path.join(k_dir, "test_fold_in.parquet"), index=False)
    tt.to_parquet(os.path.join(k_dir, "test_target.parquet"), index=False)
    
    print(f"Đã lưu hoàn tất thư mục: {k_dir}")

shutil.make_archive("/kaggle/working/few_shot_splits", 'zip', "/kaggle/working/few_shot_splits")
print("Đã nén xong!")

FileLink('few_shot_splits.zip')

Đã lưu: train.parquet
Đã lưu hoàn tất thư mục: /kaggle/working/few_shot_splits/k_1
Đã lưu hoàn tất thư mục: /kaggle/working/few_shot_splits/k_3
Đã lưu hoàn tất thư mục: /kaggle/working/few_shot_splits/k_5
Đã nén xong!


/kaggle/working/few_shot_splits.zip

In [11]:
print('Completed!')

Completed!
